# NB00 — Project Setup and Reproducibility

Creates/verifies the Drive project structure, inspects the dataset, records environment metadata, and computes a SHA-256 checksum.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

import sklearn, matplotlib, scipy
meta = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seeds": SEEDS,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS
}
with open(DATASET, "rb") as f:
    meta["dataset_sha256"] = hashlib.sha256(f.read()).hexdigest()

df = pd.read_excel(DATASET)
meta["n_rows"] = int(df.shape[0])
meta["n_columns"] = int(df.shape[1])
meta["columns"] = df.columns.tolist()
meta["class_counts"] = df[TARGET].value_counts().to_dict()

out = RESULTS / "NB00_SETUP"
out.mkdir(exist_ok=True)
with open(out/"environment_and_dataset_metadata.json","w") as f:
    json.dump(meta,f,indent=2,default=str)

print(json.dumps(meta,indent=2,default=str))


In [ ]:

# Project manifest
folders = [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]
manifest = pd.DataFrame({"folder":[str(p) for p in folders], "exists":[p.exists() for p in folders]})
manifest.to_csv(RESULTS/"NB00_SETUP"/"folder_manifest.csv",index=False)
display(manifest)
